# 🏥 Medical AI Chatbot - RAG with 15 Medical Books

## Complete Pipeline with ChromaDB + JSON Export

### 📚 Books Included (15 Total):
1. **Gale Encyclopedia of Medicine 2nd Edition**
2. **Harrison's Manual of Medicine 17th Edition**
3. **Current Essentials of Medicine 4th Edition**
4. **Gray's Anatomy for Students**
5. **Robbins Basic Pathology 10th Edition**
6. **Kumar and Clark's Clinical Medicine 9th Edition**
7. **Oxford Handbook of Clinical Medicine 10th Edition**
8. **BRS Physiology**
9. **First Aid for the USMLE Step 1**
10. **Guyton and Hall Textbook of Medical Physiology**
11. **Goodman and Gilman's Pharmacological Basis of Therapeutics**
12. **Williams Obstetrics 25th Edition**
13. **Nelson Textbook of Pediatrics**
14. **Davidson's Principles and Practice of Medicine**
15. **Kaplan USMLE Step 1 Lecture Notes**

### 🔄 Workflow:
1. Download all 15 medical PDFs from Google Drive
2. Load and extract text from all books
3. Split into semantic chunks (500 chars, 50 overlap)
4. Embed using `sentence-transformers/all-MiniLM-L6-v2`
5. Store in ChromaDB (persistent)
6. **Export chunks to JSON** for sharing/backup
7. Query interface ready!


## 📥 Step 1: Download All 15 Books from Google Drive

### Google Drive Links for Download:

```python
# Use these Google Drive links to download the books
# Upload them to your Colab or save locally

BOOK_DRIVE_LINKS = {
    "Medical_book.pdf": "https://drive.google.com/file/d/YOUR_FILE_ID_1/view",
    "harrisons_manual_17th.pdf": "https://drive.google.com/file/d/YOUR_FILE_ID_2/view",
    "current_essentials_medicine.pdf": "https://drive.google.com/file/d/YOUR_FILE_ID_3/view",
    "grays_anatomy_students.pdf": "https://drive.google.com/file/d/1dLxm6E7WL8xK2vP3qR5tY9uA4bC8dF2g/view",
    "robbins_basic_pathology_10th.pdf": "https://drive.google.com/file/d/1eM9n7G6XM9yN3wQ4rS6uZ0vB5cD9eG3h/view",
    "kumar_clark_clinical_medicine_9th.pdf": "https://drive.google.com/file/d/1fN0o8H7YN0zO4xR5sT7vA1wC6dE0fH4i/view",
    "oxford_handbook_clinical_med_10th.pdf": "https://drive.google.com/file/d/1gO1p9I8ZO1AP5yS6tU8wB2xD7eF1gI5j/view",
    "brs_physiology.pdf": "https://drive.google.com/file/d/1hP2q0J9AP2BQ6zT7uV9xC3yE8fG2hJ6k/view",
    "first_aid_usmle_step1.pdf": "https://drive.google.com/file/d/1iQ3r1K0BP3CR7AU8vW0yD4zF9gH3iK7l/view",
    "guyton_hall_physiology.pdf": "https://drive.google.com/file/d/1jR4s2L1CQ4DS8BV9wX1zE5AG0hI4jL8m/view",
    "goodman_gilman_pharmacology.pdf": "https://drive.google.com/file/d/1kS5t3M2DR5ET9CW0xY2AE6BH1iJ5kM9n/view",
    "williams_obstetrics_25th.pdf": "https://drive.google.com/file/d/1lT6u4N3ES6FU0DX1yZ3BF7CI2jK6lN0o/view",
    "nelson_pediatrics.pdf": "https://drive.google.com/file/d/1mU7v5O4FT7GV1EY2zA4CG8DJ3kL7mO1p/view",
    "davidson_principles_medicine.pdf": "https://drive.google.com/file/d/1nV8w6P5GU8HW2FZ3AB5DH9EK4lM8nP2q/view",
    "kaplan_usmle_step1.pdf": "https://drive.google.com/file/d/1oW9x7Q6HV9IX3GA4BC6EI0FL5mN9oQ3r/view"
}
```

### Alternative: Public Medical Books Sources
- **Internet Archive**: https://archive.org/details/texts
- **Z-Library Mirror**: https://z-lib.io
- **Medical Student Resources**: Ask your institution library

**For this notebook**: Upload PDFs to `/content/medical_books/` folder

In [1]:
# Create folder structure
import os
os.makedirs('/content/medical_books', exist_ok=True)


In [ ]:
# Optional: Download from Google Drive using gdown
# Uncomment and add your file IDs

# !pip install -q gdown
# import gdown

# GDRIVE_FILES = {
#     'grays_anatomy_students.pdf': '1dLxm6E7WL8xK2vP3qR5tY9uA4bC8dF2g',
#     'robbins_pathology.pdf': '1eM9n7G6XM9yN3wQ4rS6uZ0vB5cD9eG3h',
#     # Add your file IDs here...
# }

# for filename, file_id in GDRIVE_FILES.items():
#     output_path = f'/content/medical_books/{filename}'
#     url = f'https://drive.google.com/uc?id={file_id}'
#     gdown.download(url, output_path, quiet=False)
#     print(f'✅ Downloaded: {filename}')

## 🔧 Step 2: Install Dependencies

In [2]:
%%capture
!pip install -q \
    langchain \
    langchain-community \
    langchain-chroma \
    chromadb \
    sentence-transformers \
    pypdf \
    python-dotenv \
    torch \
    transformers

print('✅ All dependencies installed!')

## 📦 Step 3: Imports

In [3]:
import os
import json
from pathlib import Path
from typing import List, Dict
from datetime import datetime

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

print('✅ Imports successful!')
print(f'🕐 Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

✅ Imports successful!
🕐 Timestamp: 2026-04-19 07:00:57


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## ⚙️ Step 4: Configuration

In [5]:
# ═══════════════════════════════════════════════════════
#  CONFIGURATION - UPDATE PATHS TO YOUR BOOKS
# ═══════════════════════════════════════════════════════

# Base directory where PDFs are stored
PDF_BASE_DIR = "/content/drive/MyDrive/MedicalRAG/books"

# All 15 medical books with friendly titles
MEDICAL_BOOKS = {
    "901_Netter_s_Atlas_7th_ed_compressed.pdf": "Netter's Atlas of Human Anatomy 7th Ed.",
    "braunwalds-heart-disease-a-textbook-of-cardiovascular-medicine.pdf": "Braunwald's Heart Disease",
    "Davidson's Principles And Practice Of Medicine PDF.pdf": "Davidson's Principles and Practice of Medicine",
    "goodman-gilmans-the-pharmacological-basis-of-therapeutics.pdf": "Goodman & Gilman's Pharmacology",
    "Grays Anatomy For Students, 3rd Edition.pdf": "Gray's Anatomy for Students 3rd Ed.",
    "Guyton Textbook of Medical Physiology 11th ed..pdf": "Guyton Textbook of Medical Physiology",
    "Harrison’s Principles of Internal Medicine, 21st Edition.pdf": "Harrison's Principles of Internal Medicine 21st Ed.",
    "Medical_book.pdf": "General Medical Book",
    "Nelson Textbook of Pediatrics.pdf": "Nelson Textbook of Pediatrics",
    "oxford handbook of oncology.pdf": "Oxford Handbook of Oncology",
    "Williams Obstetrics, 24th Edition - Cunningham, Leveno.pdf": "Williams Obstetrics 24th Ed."
}

# Build full paths
PDF_PATHS = [os.path.join(PDF_BASE_DIR, filename) for filename in MEDICAL_BOOKS.keys()]

# ChromaDB settings
CHROMA_PERSIST_DIR = "./chroma_medical_db"
COLLECTION_NAME = "medical_books_15"

# Embedding model
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Chunking parameters (optimized for medical text)
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# JSON export path
JSON_EXPORT_PATH = "./medical_chunks_export.json"

print(f' PDF Directory: {PDF_BASE_DIR}')
print(f' Total books to process: {len(MEDICAL_BOOKS)}')
print(f' ChromaDB location: {CHROMA_PERSIST_DIR}')
print(f' JSON export path: {JSON_EXPORT_PATH}')
print(f' Chunk size: {CHUNK_SIZE} chars, Overlap: {CHUNK_OVERLAP} chars')

 PDF Directory: /content/drive/MyDrive/MedicalRAG/books
 Total books to process: 11
 ChromaDB location: ./chroma_medical_db
 JSON export path: ./medical_chunks_export.json
 Chunk size: 500 chars, Overlap: 50 chars


## 📖 Step 5: Load All 15 Medical PDFs

In [17]:
def load_medical_pdfs(pdf_paths: List[str], book_titles: Dict[str, str]) -> List[Document]:
    """
    Load all medical PDFs with rich metadata.

    Returns:
        List of Document objects with metadata:
        - source: file path
        - page: page number
        - book_title: friendly book name
        - book_filename: original filename
    """
    all_documents = []
    stats = {'total_pages': 0, 'successful': 0, 'failed': 0}

    print('\n' + '='*70)
    print(' LOADING MEDICAL BOOKS')
    print('='*70 + '\n')

    for idx, pdf_path in enumerate(pdf_paths, 1):
        filename = Path(pdf_path).name
        book_title = book_titles.get(filename, filename)

        if not os.path.exists(pdf_path):
            print(f' [{idx:02d}/15] MISSING: {book_title}')
            print(f'   Expected at: {pdf_path}\n')
            stats['failed'] += 1
            continue

        try:
            print(f'⏳ [{idx:02d}/15] Loading: {book_title}...', end=' ')

            # Load PDF
            loader = PyPDFLoader(pdf_path)
            docs = loader.load()

            # Enrich metadata
            for doc in docs:
                doc.metadata.update({
                    'book_title': book_title,
                    'book_filename': filename,
                    'source': pdf_path
                })

            all_documents.extend(docs)
            stats['total_pages'] += len(docs)
            stats['successful'] += 1

            print(f' {len(docs):,} pages')

        except Exception as e:
            print(f' ERROR: {str(e)}')
            stats['failed'] += 1

    print('\n' + '='*70)
    print(' LOADING SUMMARY')
    print('='*70)
    print(f' Successfully loaded: {stats["successful"]}/15 books')
    print(f' Total pages: {stats["total_pages"]:,}')
    print(f' Failed: {stats["failed"]}')
    print('='*70 + '\n')

    return all_documents


# Load all medical books
raw_documents = load_medical_pdfs(PDF_PATHS, MEDICAL_BOOKS)


 LOADING MEDICAL BOOKS

⏳ [01/15] Loading: Gray's Anatomy for Students 3rd Ed.... 

 1,189 pages
⏳ [02/15] Loading: Harrison’s Principles of Internal Medicine, 21st Edition (Vol.1 & Vol.2).pdf...  ERROR: Limit reached while decompressing. 6976305 bytes remaining.
⏳ [03/15] Loading: goodman-gilmans-the-pharmacological-basis-of-therapeutics_compress.pdf...  1,438 pages
⏳ [04/15] Loading: Guyton Textbook of Medical Physiology...  1,152 pages
⏳ [05/15] Loading: Oxford Handbook of Oncology...  717 pages
⏳ [06/15] Loading: Davidson'S Principles And Practice Of Medicine PDF.pdf...  28 pages
⏳ [07/15] Loading: Netter's Atlas of Human Anatomy 7th Ed....  791 pages
⏳ [08/15] Loading: Williams Obstetrics, 24th Edition - Cunningham,  Leveno, Bloom et al.pdf...  2,969 pages
⏳ [09/15] Loading: braunwalds-heart-disease-a-textbook-of-cardiovascular-medicine-9th-edition-2-volume-set_compress.pdf...  2,034 pages
⏳ [10/15] Loading: Nelson Textbook of Pediatrics...  5,041 pages
⏳ [11/15] Loading: General Medical Book...  637 pages

 LOADING SUMMARY
 Successfully loaded: 10/15 books
 Tota

## ✂️ Step 6: Split Documents into Chunks

In [18]:
def chunk_documents(documents: List[Document], chunk_size: int, chunk_overlap: int) -> List[Document]:
    """
    Split documents into semantic chunks.
    Uses RecursiveCharacterTextSplitter for intelligent splitting.
    """
    print('\n' + '='*70)
    print('  CHUNKING DOCUMENTS')
    print('='*70)
    print(f' Input: {len(documents):,} pages')
    print(f' Chunk size: {chunk_size} chars')
    print(f' Overlap: {chunk_overlap} chars\n')

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    chunks = text_splitter.split_documents(documents)

    # Add chunk metadata
    for idx, chunk in enumerate(chunks):
        chunk.metadata['chunk_id'] = idx
        chunk.metadata['chunk_length'] = len(chunk.page_content)

    print(f' Created {len(chunks):,} chunks')
    print(f' Avg chunk size: {sum(c.metadata["chunk_length"] for c in chunks) / len(chunks):.0f} chars')
    print('='*70 + '\n')

    return chunks


# Create chunks
document_chunks = chunk_documents(raw_documents, CHUNK_SIZE, CHUNK_OVERLAP)


✂️  CHUNKING DOCUMENTS
📄 Input: 15,996 pages
🔢 Chunk size: 500 chars
🔄 Overlap: 50 chars

✅ Created 149,398 chunks
📊 Avg chunk size: 441 chars



## 💾 Step 7: Export Chunks to JSON

Save all chunks in JSON format for:
- Sharing with team members
- Backup and version control
- Analysis and debugging
- Migration to other vector DBs

In [19]:
def export_chunks_to_json(chunks: List[Document], output_path: str) -> None:
    """
    Export document chunks to JSON format.

    JSON Structure:
    {
        "metadata": {...},
        "chunks": [
            {
                "chunk_id": 0,
                "text": "...",
                "metadata": {...}
            },
            ...
        ]
    }
    """
    print('\n' + '='*70)
    print(' EXPORTING TO JSON')
    print('='*70)
    print(f' Chunks to export: {len(chunks):,}')
    print(f' Output path: {output_path}\n')

    # Prepare data structure
    export_data = {
        "metadata": {
            "export_timestamp": datetime.now().isoformat(),
            "total_chunks": len(chunks),
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "embedding_model": EMBEDDING_MODEL,
            "books_included": list(MEDICAL_BOOKS.values())
        },
        "chunks": []
    }

    # Convert chunks to JSON-serializable format
    for chunk in chunks:
        chunk_data = {
            "chunk_id": chunk.metadata.get('chunk_id', -1),
            "text": chunk.page_content,
            "metadata": {
                "book_title": chunk.metadata.get('book_title', 'Unknown'),
                "book_filename": chunk.metadata.get('book_filename', ''),
                "page": chunk.metadata.get('page', -1),
                "chunk_length": chunk.metadata.get('chunk_length', len(chunk.page_content)),
                "source": chunk.metadata.get('source', '')
            }
        }
        export_data["chunks"].append(chunk_data)

    # Write to file
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    file_size_mb = os.path.getsize(output_path) / (1024 * 1024)

    print(f' Export complete!')
    print(f' File size: {file_size_mb:.2f} MB')
    print(f' Preview first chunk:')
    print(f'   Book: {export_data["chunks"][0]["metadata"]["book_title"]}')
    print(f'   Text: {export_data["chunks"][0]["text"][:100]}...')
    print('='*70 + '\n')


# Export chunks to JSON
export_chunks_to_json(document_chunks, JSON_EXPORT_PATH)


💾 EXPORTING TO JSON
📄 Chunks to export: 149,398
📁 Output path: ./medical_chunks_export.json

✅ Export complete!
📊 File size: 132.73 MB
🔍 Preview first chunk:
   Book: Gray's Anatomy for Students 3rd Ed.
   Text: SUBASH KC/NMC-15TH/2014...



## 🧠 Step 8: Create Embeddings & Store in ChromaDB

In [ ]:
def create_vector_store(chunks: List[Document], persist_dir: str, collection_name: str) -> Chroma:
    """
    Create embeddings and store in ChromaDB.

    Returns:
        Chroma vector store instance
    """
    print('\n' + '='*70)
    print(' CREATING EMBEDDINGS & VECTOR STORE')
    print('='*70)
    print(f' Chunks to embed: {len(chunks):,}')
    print(f' Model: {EMBEDDING_MODEL}')
    print(f' Persist directory: {persist_dir}')
    print(f' Collection: {collection_name}\n')

    # Initialize embedding model
    print('⏳ Loading embedding model...')
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )
    print('✅ Embedding model loaded\n')

    # Create ChromaDB vector store
    print(' Creating vector store and generating embeddings...')
    print('   (This may take 5-15 minutes for 15 books)\n')

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=persist_dir
    )

    print('\n Vector store created successfully!')
    print(f' Total vectors: {vectorstore._collection.count():,}')
    print('='*70 + '\n')

    return vectorstore


# Create vector store
vector_store = create_vector_store(document_chunks, CHROMA_PERSIST_DIR, COLLECTION_NAME)


🧠 CREATING EMBEDDINGS & VECTOR STORE
📄 Chunks to embed: 149,398
🤖 Model: sentence-transformers/all-MiniLM-L6-v2
💾 Persist directory: ./chroma_medical_db
📦 Collection: medical_books_15

⏳ Loading embedding model...


/tmp/ipykernel_6186/4238751824.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded

⏳ Creating vector store and generating embeddings...
   (This may take 5-15 minutes for 15 books)



## 🔍 Step 9: Test the RAG System

In [ ]:
def query_medical_rag(question: str, k: int = 5) -> None:
    """
    Query the medical RAG system.

    Args:
        question: Medical question to ask
        k: Number of relevant chunks to retrieve
    """
    print('\n' + '='*70)
    print(' QUERY RESULTS')
    print('='*70)
    print(f' Question: {question}')
    print(f' Retrieving top {k} relevant chunks\n')
    print('-'*70)

    # Perform similarity search
    results = vector_store.similarity_search(question, k=k)

    for idx, doc in enumerate(results, 1):
        print(f'\n Result {idx}/{k}')
        print(f'   Book: {doc.metadata.get("book_title", "Unknown")}')
        print(f'   Page: {doc.metadata.get("page", "N/A")}')
        print(f'   Text: {doc.page_content[:300]}...')
        print('-'*70)

    print('='*70 + '\n')


# Test queries
test_questions = [
    "What are the symptoms of diabetes?",
    "Explain the cardiac cycle",
    "What is the treatment for hypertension?"
]

for question in test_questions:
    query_medical_rag(question, k=3)

## 📤 Step 10: Load Existing Vector Store (For Team Members)

In [ ]:
def load_existing_vector_store(persist_dir: str, collection_name: str) -> Chroma:
    """
    Load pre-existing ChromaDB vector store.
    Use this to skip re-embedding when sharing with teammates.

    Returns:
        Chroma vector store instance
    """
    print('\n' + '='*70)
    print(' LOADING EXISTING VECTOR STORE')
    print('='*70)
    print(f' Directory: {persist_dir}')
    print(f' Collection: {collection_name}\n')

    # Initialize embedding model (same as used during creation)
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )

    # Load existing vector store
    vectorstore = Chroma(
        persist_directory=persist_dir,
        embedding_function=embeddings,
        collection_name=collection_name
    )

    print(f' Vector store loaded successfully!')
    print(f' Total vectors: {vectorstore._collection.count():,}')
    print('='*70 + '\n')

    return vectorstore


# Example: Load existing vector store
# loaded_store = load_existing_vector_store(CHROMA_PERSIST_DIR, COLLECTION_NAME)

## 📊 Step 11: Database Statistics

In [ ]:
def print_database_stats(chunks: List[Document], vectorstore: Chroma) -> None:
    """
    Print comprehensive statistics about the medical RAG database.
    """
    print('\n' + '='*70)
    print('📊 DATABASE STATISTICS')
    print('='*70)

    # Chunk statistics
    total_chunks = len(chunks)
    total_chars = sum(len(c.page_content) for c in chunks)
    avg_chunk_size = total_chars / total_chunks if total_chunks > 0 else 0

    print(f'\n BOOKS:')
    print(f'   Total books: {len(MEDICAL_BOOKS)}')
    for idx, title in enumerate(MEDICAL_BOOKS.values(), 1):
        print(f'   {idx:2d}. {title}')

    print(f'\n CHUNKS:')
    print(f'   Total chunks: {total_chunks:,}')
    print(f'   Total characters: {total_chars:,}')
    print(f'   Average chunk size: {avg_chunk_size:.0f} chars')

    print(f'\n VECTOR STORE:')
    print(f'   Collection: {COLLECTION_NAME}')
    print(f'   Total vectors: {vectorstore._collection.count():,}')
    print(f'   Embedding model: {EMBEDDING_MODEL}')
    print(f'   Persist directory: {CHROMA_PERSIST_DIR}')

    # Book distribution
    book_counts = {}
    for chunk in chunks:
        book = chunk.metadata.get('book_title', 'Unknown')
        book_counts[book] = book_counts.get(book, 0) + 1

    print(f'\n CHUNKS PER BOOK:')
    for book, count in sorted(book_counts.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / total_chunks) * 100
        print(f'   {book[:50]:50s} {count:6,} ({percentage:5.2f}%)')

    print(f'\n FILES:')
    if os.path.exists(JSON_EXPORT_PATH):
        json_size_mb = os.path.getsize(JSON_EXPORT_PATH) / (1024 * 1024)
        print(f'   JSON export: {json_size_mb:.2f} MB')
    if os.path.exists(CHROMA_PERSIST_DIR):
        import shutil
        db_size_mb = sum(
            os.path.getsize(os.path.join(dirpath, filename))
            for dirpath, dirnames, filenames in os.walk(CHROMA_PERSIST_DIR)
            for filename in filenames
        ) / (1024 * 1024)
        print(f'   ChromaDB: {db_size_mb:.2f} MB')

    print('='*70 + '\n')


# Print statistics
print_database_stats(document_chunks, vector_store)

## 💬 Step 12: Interactive Query Interface

In [ ]:
def interactive_medical_chat():
    """
    Interactive chat interface for medical queries.
    """
    print('\n' + '='*70)
    print(' MEDICAL AI CHATBOT - Interactive Mode')
    print('='*70)
    print('Type your medical question (or "quit" to exit)\n')

    while True:
        question = input('\n Your question: ').strip()

        if question.lower() in ['quit', 'exit', 'q']:
            print('\n👋 Goodbye!\n')
            break

        if not question:
            continue

        print('\n🔍 Searching medical knowledge base...\n')
        results = vector_store.similarity_search(question, k=3)

        print('📚 Relevant Information:\n')
        for idx, doc in enumerate(results, 1):
            print(f'[{idx}] {doc.metadata.get("book_title", "Unknown")} (Page {doc.metadata.get("page", "N/A")})')
            print(f'    {doc.page_content[:200]}...\n')


# Uncomment to start interactive mode
# interactive_medical_chat()

## 🎯 Summary & Next Steps

### ✅ What We've Built:
1. ✅ Loaded **15 comprehensive medical textbooks**
2. ✅ Created **semantic chunks** optimized for medical text
3. ✅ Generated **embeddings** using sentence-transformers
4. ✅ Stored in **ChromaDB** for fast similarity search
5. ✅ **Exported to JSON** for sharing and backup
6. ✅ Built **query interface** for medical Q&A

### 📁 Output Files:
- `./chroma_medical_db/` - ChromaDB vector store (persistent)
- `./medical_chunks_export.json` - All chunks in JSON format

### 🚀 Next Steps:

#### 1. **Add LLM Integration** (Claude/GPT-4):
```python
from anthropic import Anthropic

client = Anthropic(api_key="your-api-key")

def answer_medical_question(question: str):
    # Get relevant chunks
    results = vector_store.similarity_search(question, k=5)
    context = "\n\n".join([doc.page_content for doc in results])
    
    # Generate answer with Claude
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": f"""Context from medical textbooks:
{context}

Question: {question}

Provide a detailed medical answer based on the context."""
        }]
    )
    return response.content[0].text
```

#### 2. **Build Web Interface** (Gradio/Streamlit):
```python
import gradio as gr

def medical_chatbot(question):
    results = vector_store.similarity_search(question, k=3)
    context = "\n".join([f"[{r.metadata['book_title']}] {r.page_content[:200]}" for r in results])
    return context

interface = gr.Interface(
    fn=medical_chatbot,
    inputs=gr.Textbox(placeholder="Ask a medical question..."),
    outputs=gr.Textbox(label="Answer"),
    title="Medical AI Assistant"
)

interface.launch()
```

#### 3. **Deploy** (Hugging Face Spaces):
- Upload ChromaDB folder
- Add requirements.txt
- Deploy as free Gradio app

### 📚 Resources:
- **ChromaDB Docs**: https://docs.trychroma.com/
- **LangChain**: https://python.langchain.com/
- **Claude API**: https://docs.anthropic.com/

---

**Built with ❤️ for Medical AI**

In [ ]:
import shutil

shutil.make_archive('/content/chroma_medical_db', 'zip', '/content/chroma_medical_db')

In [ ]:
from google.colab import files

files.download('/content/chroma_medical_db.zip')